# 🤖 Feature Engineering & Model v0
**World Cup 2026 Predictor** · Building pre-match features from historical data and training the first prediction model.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

PATH = "../data/raw"
matches  = pd.read_csv(PATH + "/historical/matches.csv")
fixture  = pd.read_csv(PATH + "/fixture_2026/fixture_2026.csv")
squads26 = pd.read_csv(PATH + "/squads_2026/squads_2026.csv")

## 1. Preparation — men's matches, heir nations and naming fixes

In [2]:
# Men's World Cups only
matches_m = matches[matches["tournament_name"].str.contains("Men's")].copy()
matches_m["year"] = matches_m["tournament_name"].str[:4].astype(int)

# Map defunct nations to their FIFA heirs (so Germany inherits West Germany's history, etc.)
HEIR_NATIONS = {
    "West Germany": "Germany",
    "Soviet Union": "Russia",
    "Yugoslavia": "Serbia",
    "Serbia and Montenegro": "Serbia",
    "Czechoslovakia": "Czech Republic",
    "Zaire": "DR Congo",
    "Dutch East Indies": "Indonesia",
}
matches_m["home_team_name"] = matches_m["home_team_name"].replace(HEIR_NATIONS)
matches_m["away_team_name"] = matches_m["away_team_name"].replace(HEIR_NATIONS)

# Standardize 2026 fixture naming (same fix as notebook 01)
name_map = {"USA": "United States", "Bosnia & Herzegovina": "Bosnia and Herzegovina"}
fixture["team1"] = fixture["team1"].replace(name_map)
fixture["team2"] = fixture["team2"].replace(name_map)

print("Matches:", len(matches_m), "| Years:", matches_m["year"].min(), "-", matches_m["year"].max())

Matches: 964 | Years: 1930 - 2022


## 2. Long format — one row per team per match

In [3]:
# Home perspective
home = matches_m[["year", "match_id", "home_team_name", "away_team_name",
                  "home_team_score", "away_team_score", "home_team_win", "draw"]].copy()
home.columns = ["year", "match_id", "team", "opponent", "goals_for", "goals_against", "won", "draw"]

# Away perspective
away = matches_m[["year", "match_id", "away_team_name", "home_team_name",
                  "away_team_score", "home_team_score", "away_team_win", "draw"]].copy()
away.columns = ["year", "match_id", "team", "opponent", "goals_for", "goals_against", "won", "draw"]

team_matches = pd.concat([home, away], ignore_index=True)
print(team_matches.shape)
team_matches[team_matches["team"] == "Brazil"].head()

(1928, 8)


,year,match_id,team,opponent,goals_for,goals_against,won,draw
11,1930,M-1930-12,Brazil,Bolivia,4,0,1,0
40,1938,M-1938-06,Brazil,Poland,6,5,1,0
44,1938,M-1938-10,Brazil,Czech Republic,1,1,0,1
48,1938,M-1938-14,Brazil,Czech Republic,2,1,1,0
51,1938,M-1938-17,Brazil,Sweden,4,2,1,0


## 3. Pre-match feature builder (no data leakage: only history BEFORE the given year)

In [4]:
def team_features(team, year, prefix):
    """Historical features for a team using ONLY matches played BEFORE `year`."""
    past = team_matches[(team_matches["team"] == team) & (team_matches["year"] < year)]

    if len(past) == 0:  # World Cup debutant
        return {
            f"{prefix}_wc_played": 0, f"{prefix}_matches": 0,
            f"{prefix}_win_rate": 0.0, f"{prefix}_draw_rate": 0.0,
            f"{prefix}_goals_for": 0.0, f"{prefix}_goals_against": 0.0,
            f"{prefix}_recent_form": 0.0,
        }

    # Recent form: win rate in the team's last 2 World Cups
    last_two = sorted(past["year"].unique())[-2:]
    recent = past[past["year"].isin(last_two)]

    return {
        f"{prefix}_wc_played": past["year"].nunique(),
        f"{prefix}_matches": len(past),
        f"{prefix}_win_rate": past["won"].mean(),
        f"{prefix}_draw_rate": past["draw"].mean(),
        f"{prefix}_goals_for": past["goals_for"].mean(),
        f"{prefix}_goals_against": past["goals_against"].mean(),
        f"{prefix}_recent_form": recent["won"].mean(),
    }

# Quick sanity check: Brazil before 2002 vs before 1962
print(team_features("Brazil", 2002, "home"))
print(team_features("Brazil", 1962, "home"))

{'home_wc_played': 16, 'home_matches': 80, 'home_win_rate': np.float64(0.6875), 'home_draw_rate': np.float64(0.1375), 'home_goals_for': np.float64(2.1625), 'home_goals_against': np.float64(0.975), 'home_recent_form': np.float64(0.7857142857142857)}
{'home_wc_played': 6, 'home_matches': 23, 'home_win_rate': np.float64(0.6086956521739131), 'home_draw_rate': np.float64(0.17391304347826086), 'home_goals_for': np.float64(2.869565217391304), 'home_goals_against': np.float64(1.3478260869565217), 'home_recent_form': np.float64(0.6666666666666666)}


## 4. Training dataset — group-stage matches from 1962 onwards

In [6]:
# Host country per tournament (for the is_host feature)
tournaments = pd.read_csv(PATH + "/historical/tournaments.csv")
tournaments_m = tournaments[tournaments["tournament_name"].str.contains("Men's")].copy()
tournaments_m["year"] = tournaments_m["tournament_name"].str[:4].astype(int)
host_by_year = tournaments_m.set_index("year")["host_country"].to_dict()

# Group-stage matches from 1962 (the EDA decision)
train_matches = matches_m[(matches_m["year"] >= 1962) & (matches_m["group_stage"] == 1)].copy()
print("Training matches:", len(train_matches))

rows = []
for _, m in train_matches.iterrows():
    row = {"year": m["year"], "home_team": m["home_team_name"], "away_team": m["away_team_name"]}

    # Historical features for both teams (only data before this tournament)
    row.update(team_features(m["home_team_name"], m["year"], "home"))
    row.update(team_features(m["away_team_name"], m["year"], "away"))

    # Host feature
    host = host_by_year.get(m["year"], "")
    row["home_is_host"] = int(m["home_team_name"] == host)
    row["away_is_host"] = int(m["away_team_name"] == host)

    # Target: 0 = home win, 1 = draw, 2 = away win
    row["target"] = 0 if m["home_team_win"] == 1 else (1 if m["draw"] == 1 else 2)
    rows.append(row)

dataset = pd.DataFrame(rows)
print(dataset.shape)
dataset.head()

Training matches: 636
(636, 20)


,year,home_team,away_team,home_wc_played,home_matches,home_win_rate,home_draw_rate,home_goals_for,home_goals_against,home_recent_form,away_wc_played,away_matches,away_win_rate,away_draw_rate,away_goals_for,away_goals_against,away_recent_form,home_is_host,away_is_host,target
0,1962,Uruguay,Colombia,3,13,0.769231,0.076923,3.538462,1.307692,0.666667,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0
1,1962,Chile,Switzerland,2,6,0.500000,0.000000,1.666667,1.500000,0.500000,4,12,0.416667,0.166667,2.083333,2.250000,0.428571,1,0,0
2,1962,Brazil,Mexico,6,23,0.608696,0.173913,2.869565,1.347826,0.666667,4,11,0.000000,0.090909,0.818182,3.545455,0.000000,0,0,0
3,1962,Argentina,Bulgaria,3,9,0.555556,0.000000,2.777778,2.444444,0.250000,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0
4,1962,Russia,Serbia,1,5,0.400000,0.200000,1.000000,1.200000,0.400000,4,13,0.461538,0.230769,1.769231,1.538462,0.285714,0,0,0


In [7]:
# Class balance (should be ~51 / 24 / 24 as we saw in the EDA)
print(dataset["target"].value_counts(normalize=True).round(3) * 100)

# Difference features: often more informative than absolute values
dataset["diff_win_rate"]  = dataset["home_win_rate"]  - dataset["away_win_rate"]
dataset["diff_goals_for"] = dataset["home_goals_for"] - dataset["away_goals_for"]
dataset["diff_form"]      = dataset["home_recent_form"] - dataset["away_recent_form"]
dataset["diff_exp"]       = dataset["home_wc_played"]  - dataset["away_wc_played"]

target
0    47.5
2    27.4
1    25.2
Name: proportion, dtype: float64


## 5. Model v0 — temporal split, training and evaluation

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

FEATURES = [
    "home_wc_played", "home_win_rate", "home_draw_rate", "home_goals_for",
    "home_goals_against", "home_recent_form", "home_is_host",
    "away_wc_played", "away_win_rate", "away_draw_rate", "away_goals_for",
    "away_goals_against", "away_recent_form", "away_is_host",
    "diff_win_rate", "diff_goals_for", "diff_form", "diff_exp",
]

# Temporal split: train up to 2014, test on 2018 + 2022 (simulates predicting future tournaments)
train = dataset[dataset["year"] <= 2014]
test  = dataset[dataset["year"] >= 2018]

X_train, y_train = train[FEATURES], train["target"]
X_test,  y_test  = test[FEATURES],  test["target"]
print(f"Train: {len(train)} matches (1962-2014) | Test: {len(test)} matches (2018-2022)")

# Scale (needed for Logistic Regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

Train: 540 matches (1962-2014) | Test: 96 matches (2018-2022)


In [9]:
# Baseline: always predict "home win" (class 0)
baseline_acc = (y_test == 0).mean()
print(f"BASELINE (always home win): {baseline_acc*100:.1f}%\n")

# Logistic Regression
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_s, y_train)
pred_lr = logreg.predict(X_test_s)
print(f"Logistic Regression: {accuracy_score(y_test, pred_lr)*100:.1f}%")

# Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print(f"Random Forest:       {accuracy_score(y_test, pred_rf)*100:.1f}%")

BASELINE (always home win): 37.5%

Logistic Regression: 47.9%
Random Forest:       50.0%


In [10]:
print(classification_report(y_test, pred_lr, target_names=["Home win", "Draw", "Away win"]))
print(confusion_matrix(y_test, pred_lr))

              precision    recall  f1-score   support

    Home win       0.46      0.72      0.56        36
        Draw       0.00      0.00      0.00        19
    Away win       0.54      0.49      0.51        41

    accuracy                           0.48        96
   macro avg       0.33      0.40      0.36        96
weighted avg       0.40      0.48      0.43        96

[[26  0 10]
 [12  0  7]
 [19  2 20]]


**v0 results:** RF 50.0% vs 37.5% baseline (+12.5 pts) on 2018-2022 test.
Known issue: the model never predicts draws (minority class, as anticipated in the EDA).
v1 improvements: class_weight, probability thresholds, Elo ratings, squad features.

In [11]:
HOSTS_2026 = {"United States", "Mexico", "Canada"}

# Group-stage matches with confirmed teams
groups_2026 = fixture[fixture["group"].notna()].copy()

rows_2026 = []
for _, m in groups_2026.iterrows():
    row = {"round": m["round"], "date": m["date"], "group": m["group"],
           "home_team": m["team1"], "away_team": m["team2"]}
    row.update(team_features(m["team1"], 2026, "home"))
    row.update(team_features(m["team2"], 2026, "away"))
    row["home_is_host"] = int(m["team1"] in HOSTS_2026)
    row["away_is_host"] = int(m["team2"] in HOSTS_2026)
    rows_2026.append(row)

pred_2026 = pd.DataFrame(rows_2026)
pred_2026["diff_win_rate"]  = pred_2026["home_win_rate"]  - pred_2026["away_win_rate"]
pred_2026["diff_goals_for"] = pred_2026["home_goals_for"] - pred_2026["away_goals_for"]
pred_2026["diff_form"]      = pred_2026["home_recent_form"] - pred_2026["away_recent_form"]
pred_2026["diff_exp"]       = pred_2026["home_wc_played"]  - pred_2026["away_wc_played"]

# Probabilities with the best model (Random Forest)
probs = rf.predict_proba(pred_2026[FEATURES])
pred_2026["p_home_win"] = (probs[:, 0] * 100).round(1)
pred_2026["p_draw"]     = (probs[:, 1] * 100).round(1)
pred_2026["p_away_win"] = (probs[:, 2] * 100).round(1)
pred_2026["prediction"] = np.array(["Home win", "Draw", "Away win"])[probs.argmax(axis=1)]

print(pred_2026.shape)

(72, 29)


In [12]:
matchday1 = pred_2026[pred_2026["round"] == "Matchday 1"][
    ["date", "group", "home_team", "away_team", "p_home_win", "p_draw", "p_away_win", "prediction"]
].sort_values("date")

matchday1

,date,group,home_team,away_team,p_home_win,p_draw,p_away_win,prediction
0,2026-06-11,Group A,Mexico,South Africa,56.1,22.5,21.4,Home win
1,2026-06-11,Group A,South Korea,Czech Republic,33.4,27.0,39.6,Away win
